# Phase 3: The Final Ensemble — Where All Roads Lead

If Notebooks 1–4 in each pipeline were separate specialists gathering evidence, this notebook is the detective's briefing room where all their reports get laid out on the table and cross-examined together. It **does not collect new data** — it consumes the *already-processed outputs* of the Kepler and TESS pipelines (`1d_X_data.npy`, `2d_X.npy`, and their label files) and trains a **three-branch stacked ensemble**: one branch looks at raw brightness-over-time, one looks at raw pixel imagery, and one looks at classical astronomical transit statistics. Their combined opinions are then fed into a final "meta" model that makes the ultimate call.

This is a genuinely more sophisticated architecture than anything in the earlier notebooks — it's less "train a CNN" and more "build a small committee of specialists and train a judge to weigh their testimony."

---

## The Big Picture (Read This First)

Before diving into each section, here's the mental model: this pipeline builds **three independent classifiers ("branches")**, each trained on a *different representation* of the same underlying stars, and then trains a **fourth, "meta" classifier** on top of the first three's predictions. This general technique is called **stacked generalization (stacking)** — instead of picking one model type and hoping it's the best, you let several different, imperfect models each contribute their own signal, and let a final model learn how much to trust each one.

The three branches, and what each one "sees":

| Branch | Data source | What it looks at | Feature extractor | Final classifier |
|---|---|---|---|---|
| **1 — Temporal** | `1d_X_data.npy` | 1D flux-over-time sequences | Conv1D + LSTM | Random Forest |
| **2 — Spatial** | `2d_X.npy` | 2D pixel images per timestep | TimeDistributed Conv2D + LSTM | XGBoost |
| **3 — BLS** | Derived from Branch 1's data | Hand-engineered transit statistics | *(none — features are already numeric)* | Random Forest **and** XGBoost, averaged |

Notice that Branches 1 and 2 use **deep learning as a feature extractor**, not as the final decision-maker — this is a subtle but important design choice. The neural network learns a compact numeric summary ("embedding") of the raw sequence/image, and a classical ML model (Random Forest or XGBoost) makes the actual yes/no call based on those learned features. This hybrid approach often outperforms using the neural network's own output layer directly, especially on smaller datasets, because tree-based models tend to be more data-efficient and less prone to overfitting than a neural net's final dense/sigmoid layer.

---

## Data Loading (`load_data`)

**What it does:** Loads four `.npy` files that were saved by earlier notebooks:
- `1d_X_data.npy` / `1d_y_labels.npy` — the resampled 1D flux sequences and labels, produced at the very end of `02_d_CNN_LSTM.ipynb` (the TESS CNN/LSTM notebook).
- `2d_X.npy` / `2d_y.npy` — the resized pixel-image tensors and labels, produced at the end of `01_dd_CNN_LSTM.ipynb` (the Kepler CNN notebook).

**Why this matters architecturally:** this is the literal seam where the two earlier pipelines' outputs get stitched together into one unified dataset. It's why this notebook is the "father" — it's the only one that reaches back and pulls in artifacts from *both* the Kepler and TESS branches simultaneously.

**Defensive details worth noting:**
- It checks all four files exist up front with a clear `FileNotFoundError` rather than crashing cryptically deep inside a later function — fail fast, fail clearly.
- It **truncates all four arrays to the shortest one's length** (`min_len = min(...)`). This is a pragmatic (if slightly blunt) way of handling the fact that the 1D and 2D pipelines were run independently and likely don't have the exact same number of samples or even represent the *same* stars in the *same* order — a real limitation worth flagging, since this assumes sample `i` in the 1D array corresponds meaningfully to sample `i` in the 2D array, which isn't guaranteed unless both pipelines were run over an identical, ordered ID list.
- It defaults to using the **1D labels** as the source of truth, printing a warning if the 1D and 2D label arrays disagree — a good instinct (surface the discrepancy rather than silently picking one), though the warning doesn't currently say *how much* they disagree, just that they do.

---

## BLS Feature Extraction (`extract_bls_features`)

This is the most scientifically distinct part of the whole notebook — a **completely different, non-deep-learning approach to transit detection**, running in parallel to the neural network branches.

**What is BLS?** The **Box Least Squares (BLS) algorithm** is a decades-old, purpose-built statistical method for detecting periodic, box-shaped dips in brightness — which is *exactly* what a planetary transit looks like (a sharp dip, a flat bottom, a sharp return to normal, repeating on a fixed period). Unlike a CNN, which has to *learn* what a transit looks like from examples, BLS is a direct search: it tries many candidate periods and transit durations, and for each one, checks "if I fit a box-shaped dip with this period/duration, how much does it improve the fit compared to no dip at all?" The best-fitting combination is returned as the most likely transit signal.

**The Algorithm, step by step, per star:**
1. Pull out that star's flux sequence and build a matching time array (using a synthetic cadence if no real timestamps were provided — `time = np.arange(T) * cadence`).
2. **Mask out non-finite values** (`np.isfinite(flux)`) — BLS can't handle `NaN`s gracefully, so they're stripped before fitting.
3. **Bail out safely** if fewer than 10 valid points remain, returning a placeholder feature vector of eight zeros rather than crashing the whole batch over one bad star.
4. **Run `astropy.timeseries.BoxLeastSquares`**, searching across **500 candidate periods** linearly spaced between 0.5 and 10.0 days, with a fixed trial transit duration of 0.1 days. This is a computationally meaningful step — for every one of the 500 candidate periods, BLS is essentially phase-folding the light curve and testing how well a box fits — repeated for every star in the dataset.
5. **Extract the single best-fitting result** (`results[np.argmax(results.power)]`) — the period/duration/depth combination that best explains the data as a periodic transit.
6. **Assemble an 8-feature summary vector per star:**
   - `period` — how often the (candidate) transit repeats, in days.
   - `duration` — how long each transit event lasts.
   - `depth` — how much the brightness drops during a transit (directly related to how big the planet is relative to its star).
   - `snr` (signal-to-noise ratio) — the best period's BLS power divided by the *median* power across all 500 candidate periods; a high value means this period stood out dramatically from the noise floor rather than being a marginal, coincidental fit.
   - `epoch` — the time of a specific reference transit, used to phase-fold future observations.
   - `n_transits` — how many complete transit cycles fit within the total observed time baseline (`total_time / period`) — a rough estimate of how many independent chances there were to observe this signal.
   - `odd_even_ratio` — currently **hardcoded to `1.0`** rather than actually computed. In real transit vetting, comparing the depth of odd-numbered vs. even-numbered transits is a classic technique for catching **false positives caused by eclipsing binary stars** (which tend to produce alternating deep/shallow dips, unlike a genuine planet, which produces uniform dips). This being a placeholder is worth flagging as a known simplification/future improvement.
   - `best.power` — the raw BLS periodogram power at the best-fitting period, a general measure of how confidently BLS identified this signal.
7. **Wrap any per-star failure in a bare `except:`**, falling back to the same eight-zeros placeholder — ensuring one problematic star's BLS fit failing doesn't take down the entire feature-extraction loop for the whole dataset.

**Why this branch matters:** BLS features encode *domain expertise* directly — decades of astronomical transit-detection knowledge distilled into eight numbers — rather than requiring a neural network to rediscover "periodicity" and "box shape" from scratch. It's a fundamentally different kind of signal from what Branches 1 and 2 provide, which is exactly why stacking it in adds genuine diversity to the ensemble rather than redundancy.

---

## Preprocessing (`preprocess_light_curves`, `preprocess_tpfs`)

Two small but important normalization functions, both doing the same conceptual thing (z-score standardization) on different-shaped data:

- **`preprocess_light_curves`**: normalizes each 1D flux sequence *individually*, computing mean/std along the time axis (`axis=1`) per sample. This means every star's light curve is centered and scaled relative to *its own* statistics, not the dataset's — important because different stars have wildly different absolute brightness levels, and what matters for detection is the *shape* of the variation, not its absolute scale.
- **`preprocess_tpfs`**: same idea, but for the pixel-image data, computing mean/std across the time, height, and width axes (`axis=(1,2,3)`) — i.e., per-sample (per-star) normalization across the *entire* pixel cube, not per-pixel or per-frame.

Both add a small epsilon (`1e-8`) to the denominator, the standard guard against division-by-zero for a hypothetically perfectly flat/constant sample.

---

## Building the Feature Extractors (the "Backbones")

This is where the two deep learning branches get their neural network architecture defined — but crucially, **as feature extractors (backbones), not full classifiers**. Notice both functions return a `Model` whose *output* is a learned feature vector, not a final prediction.

### `build_temporal_feature_extractor` (feeds Branch 1)
- Input: the 1D flux sequences.
- `Conv1D(64) → MaxPooling1D(2) → Conv1D(128) → MaxPooling1D(2)`: two rounds of convolution + downsampling, extracting increasingly abstract local patterns from the flux sequence (edges, dips, local shapes) while progressively shrinking the sequence length.
- `LSTM(128, return_sequences=False)`: takes the sequence of extracted conv features and compresses the *entire* sequence into a single 128-dimensional summary vector, capturing how those local patterns relate to each other across time — similar in spirit to the CNN+LSTM from `02_d_CNN_LSTM.ipynb`, but here explicitly built as a headless backbone rather than an end-to-end classifier.

### `build_spatial_feature_extractor` (feeds Branch 2)
This one's a bit more architecturally interesting because it has to handle a genuinely 3D-shaped input: *(time, height, width)* — a whole **sequence of images** per star, not just one image.
- **`Reshape`** adds an explicit channel dimension so each frame becomes `(height, width, 1)`, the format `Conv2D` expects.
- **`TimeDistributed(Conv2D(...))`** — this is the key trick: `TimeDistributed` applies the *same* Conv2D layer independently to every timestep/frame in the sequence, sharing weights across time. Three stacked TimeDistributed Conv2D layers (32 → 64 → 128 filters) progressively extract richer spatial features from each individual frame.
- **Deliberately no `MaxPooling2D`** anywhere in this backbone — the code comment explicitly notes this is intentional ("safe spatial CNN (no MaxPooling)"), because pooling layers can throw dimension errors on small or irregularly-shaped spatial inputs (recall these images are as small as 8×7 pixels from the Kepler pipeline). Instead...
- **`TimeDistributed(GlobalAveragePooling2D())`** collapses each frame's entire spatial feature map down to a single 128-length vector by averaging — this sidesteps the small-image dimension problem entirely, since global average pooling works on *any* spatial size without needing exact divisibility.
- **`LSTM(256, return_sequences=False)`** then does for the spatial branch what the temporal branch's LSTM does for flux: compress the whole time sequence of per-frame feature vectors into one 256-dimensional summary that captures how the star's *appearance* (not just its brightness) evolves over time — genuinely richer information than a plain brightness curve, since it can in principle pick up on things like how the flux is spatially distributed across the aperture as a transit occurs.

---

## Training Each Branch

Each `train_branchN` function follows a shared two-stage pattern: **(a)** train a neural network end-to-end (backbone + a temporary sigmoid head) just to teach the backbone to extract *useful* features, then **(b)** throw away the temporary head and use the backbone's learned features to train a completely separate classical ML model, which becomes the branch's actual final predictor.

### `train_branch1` (Temporal)
- Attaches a temporary `Dense(1, sigmoid)` head onto the temporal backbone, compiles with `binary_crossentropy`, and trains for **26 epochs** (`batch_size=64`) — this training phase exists purely to shape the backbone's internal representations to be transit-discriminative, even though this particular head gets discarded afterward.
- **Saves the backbone itself** as `best_1d_trained_model.keras` — note this *overwrites* the file of the same name produced by `02_d_CNN_LSTM.ipynb`, so after this notebook runs, that filename refers to this ensemble's temporal backbone, not the standalone TESS CNN/LSTM model from before.
- Re-wraps the backbone as a standalone `feature_extractor` model, runs training and validation data through it (`.predict()`) to get their 128-dim embeddings.
- Trains a **Random Forest** (`n_estimators=100`) directly on those embeddings — this is now Branch 1's actual decision-maker — and saves it as `branch1_rf.joblib`.

### `train_branch2` (Spatial)
- Structurally identical pattern to Branch 1, but using the spatial backbone, a smaller `batch_size=32` (sensible given the heavier per-sample compute cost of processing image sequences vs. plain 1D sequences), and saving the backbone as `best_2D-trained_model.keras` (again overwriting the Kepler pipeline's own file of that name).
- The classical model here is **XGBoost** rather than Random Forest — a deliberate choice to diversify *which kind* of classical model backs each branch, which tends to help an ensemble more than using the same classical model type everywhere.
- Saved as `branch2_xgb.joblib`.

### `train_branch3` (BLS)
- The odd one out — **no neural network at all**, because the BLS features are already a small, dense, meaningful numeric vector (8 features) rather than a raw sequence or image needing feature extraction.
- Trains **both** a Random Forest *and* an XGBoost classifier directly on the (scaled) BLS features, saving each independently (`branch3_rf.joblib`, `branch3_xgb.joblib`).
- Notably, this branch's *final* prediction later gets computed as the **average of these two models' probabilities** — effectively a tiny two-model ensemble nested inside the larger three-branch ensemble.

---

## The Meta-Classifier (`train_meta_classifier`) — The Final Judge

This is the heart of the "stacking" concept, and arguably the single most important function in the whole notebook.

**What goes in:** for every star in the *validation* set, six numbers — each branch's predicted probability (`P1`, `P2`, `P3`) *plus* a duplicate "confidence" value (`C1`, `C2`, `C3`, which here are literally identical copies of `P1`/`P2`/`P3` — a bit of redundancy in the current implementation, though presumably a placeholder for a more nuanced confidence metric down the line, like prediction variance or model agreement).

**What happens:** these six columns are stacked together (`np.column_stack`) into a small new feature matrix, `X_meta` — meaning the meta-classifier never sees raw flux, raw pixels, or raw BLS features at all. It only ever sees *"branch 1 said 73% confident, branch 2 said 61% confident, branch 3 said 88% confident"* and learns, from real outcomes, how to weigh and combine those opinions into a final, hopefully more accurate verdict than any single branch alone.

**The model itself:** an XGBoost classifier with `n_estimators=500` and `learning_rate=0.05` — deliberately more conservative (more trees, smaller learning rate) than the individual branch classifiers, since it's working with a much smaller, denser feature space and benefits from more careful, gradual learning.

**Note on validation methodology:** the meta-classifier is trained and early-stopped using the *same* validation set (`X_meta`, `y_val`) it's evaluated against (`eval_set=[(X_meta, y_val)]`). This is a meaningful methodological wrinkle worth being aware of — in a stricter setup, you'd typically want a separate held-out set for training the meta-classifier versus one for final evaluation, since training and validating on the same data risks the meta-model's reported performance being optimistic. As currently written, this reuse means the meta-classifier's `early_stopping_rounds=50` is effectively stopping based on the very data it's fitting.

**Output:** saved as `meta_xgb.joblib` — the final piece of the puzzle, and the model that would actually be used at prediction time by feeding it the three upstream branches' outputs.

---

## `main()` — The Whole Pipeline, End to End

Putting it all together, here's the full sequence of events when this notebook runs:

1. **Load** the merged 1D + 2D dataset (`load_data()`).
2. **Extract BLS features** for every star (`extract_bls_features()`) — computed *before* the neural preprocessing/normalization, since BLS wants physically meaningful flux values, not z-scored ones.
3. **Normalize** the light curves and TPF image cubes (`preprocess_light_curves`, `preprocess_tpfs`).
4. **Scale the BLS features** with a `StandardScaler` (standard for tree-based models too, though less critical than for neural nets — still good practice), saving the fitted scaler (`bls_scaler.joblib`) so any *future* star can be scaled identically before being fed to Branch 3 at prediction time.
5. **Split into train/validation** (80/20) with `stratify=labels` — ensuring the relatively rare positive-transit class is represented proportionally in both splits, which matters a lot given how class-imbalanced transit detection tends to be. Critically, this is a **single shared split** — light curves, TPFs, and BLS features are split together using the same random indices, keeping all three views of each star aligned across train/val.
6. **Train Branch 1**, generate its validation-set probability predictions (`P1_val`).
7. **Train Branch 2**, generate its validation-set probability predictions (`P2_val`).
8. **Train Branch 3** (both RF and XGB), and average their two probability outputs into `P3_val`.
9. **Train the meta-classifier** on all three branches' validation predictions plus the true validation labels.
10. **Print a summary manifest** of every file this run produced — a nice touch for reproducibility, giving you an explicit checklist of exactly what artifacts now exist on disk.

**Full list of files this notebook produces:**
- `best_1d_trained_model.keras` — Branch 1's temporal backbone
- `best_2D-trained_model.keras` — Branch 2's spatial backbone
- `branch1_rf.joblib` — Branch 1's Random Forest head
- `branch2_xgb.joblib` — Branch 2's XGBoost head
- `branch3_rf.joblib` / `branch3_xgb.joblib` — Branch 3's dual classical models
- `bls_scaler.joblib` — the fitted scaler for BLS features
- `meta_xgb.joblib` — the final stacking meta-classifier

---

## Why This Notebook Really Is "The Father"

Every earlier notebook in both pipelines (Kepler `01_*` and TESS `02_*`) exists to produce exactly one thing this notebook needs: labeled, preprocessed `1d_X_data.npy`/`2d_X.npy` arrays. This notebook is the only place where:
- Both pipelines' outputs are used **together**, in the same training run.
- Deep learning and classical ML are combined in a **principled hybrid** (neural feature extraction + tree-based classification).
- A **domain-expert, non-ML technique** (BLS) sits alongside learned features as an equal contributor.
- A **meta-model learns how to trust each of its "children"** — which is a fittingly literal way for a "father" notebook to behave.

*In short: Notebooks 1–4 each raised one specialist. This notebook is the family reunion where all three specialists compare notes, and a very serious-looking XGBoost model sits at the head of the table deciding whose opinion actually mattered.*

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib
import os
from astropy.timeseries import BoxLeastSquares
from astropy import units as u
import warnings

warnings.filterwarnings('ignore')

In [ ]:
def load_data():
    required = ['1d_X_data.npy', '1d_y_labels.npy', '2d_X.npy', '2d_y.npy']
    for f in required:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Required data file '{f}' not found.")

    print("Loading data...")
    light_curves = np.load('1d_X_data.npy')
    labels_1d = np.load('1d_y_labels.npy')
    tpfs = np.load('2d_X.npy')
    labels_2d = np.load('2d_y.npy')

    min_len = min(light_curves.shape[0], labels_1d.shape[0], tpfs.shape[0], labels_2d.shape[0])
    light_curves = light_curves[:min_len]
    labels_1d = labels_1d[:min_len]
    tpfs = tpfs[:min_len]
    labels_2d = labels_2d[:min_len]
    labels = labels_1d
    if not np.array_equal(labels_1d, labels_2d):
        print("Warning: Labels from 1D and 2D differ. Using 1D labels.")

    return light_curves, tpfs, labels

In [ ]:
def extract_bls_features(light_curves, time=None, cadence=1.0):
    N, T, _ = light_curves.shape
    if time is None:
        time = np.arange(T) * cadence
    elif len(time) != T:
        raise ValueError("Time array length must match time_steps.")

    features_list = []
    print("Extracting BLS features...")
    for i in range(N):
        flux = light_curves[i, :, 0]
        mask = np.isfinite(flux)
        t = time[mask]
        f = flux[mask]
        if len(t) < 10:
            features_list.append([0.0]*8)
            continue

        try:
            bls = BoxLeastSquares(t * u.day, f)
            periods = np.linspace(0.5, 10.0, 500)
            results = bls.power(periods * u.day, duration=0.1 * u.day)
            best = results[np.argmax(results.power)]

            period = best.period.value
            duration = best.duration.value
            depth = best.depth
            median_power = np.median(results.power)
            snr = best.power / median_power if median_power > 0 else 0
            epoch = best.transit_time.value
            total_time = t[-1] - t[0]
            n_transits = int(np.floor(total_time / period)) if period > 0 else 0
            odd_even_ratio = 1.0
            features_list.append([period, duration, depth, snr, epoch, n_transits, odd_even_ratio, best.power])
        except:
            features_list.append([0.0]*8)

    return np.array(features_list)

In [ ]:
def preprocess_light_curves(lc):
    mean = np.mean(lc, axis=1, keepdims=True)
    std = np.std(lc, axis=1, keepdims=True) + 1e-8
    return (lc - mean) / std

In [ ]:
def preprocess_tpfs(tpfs):
    mean = np.mean(tpfs, axis=(1,2,3), keepdims=True)
    std = np.std(tpfs, axis=(1,2,3), keepdims=True) + 1e-8
    return (tpfs - mean) / std

In [ ]:
def build_temporal_feature_extractor(input_shape):
    inputs = layers.Input(shape=input_shape)
    c = layers.Conv1D(64, 5, padding='same', activation='relu')(inputs)
    c = layers.MaxPooling1D(2)(c)
    c = layers.Conv1D(128, 5, padding='same', activation='relu')(c)
    c = layers.MaxPooling1D(2)(c)
    lstm_out = layers.LSTM(128, return_sequences=False)(c)
    return models.Model(inputs=inputs, outputs=lstm_out)

In [ ]:
def build_spatial_feature_extractor(input_shape):
    inputs = layers.Input(shape=input_shape)
    # Add channel dimension
    x = layers.Reshape((input_shape[0], input_shape[1], input_shape[2], 1))(inputs)

    # Conv layers with padding='same' (no pooling)
    x = layers.TimeDistributed(layers.Conv2D(32, 3, padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.Conv2D(64, 3, padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.Conv2D(128, 3, padding='same', activation='relu'))(x)

    # Global average pooling over spatial dimensions → (time, 128)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)

    # LSTM over time
    lstm_out = layers.LSTM(256, return_sequences=False)(x)
    return models.Model(inputs=inputs, outputs=lstm_out)

In [ ]:
def train_branch1(X_train, y_train, X_val, y_val, input_shape, save_path='best_1d_trained_model.keras'):
    print("===== Branch 1 (Temporal) =====")
    backbone = build_temporal_feature_extractor(input_shape)
    features = backbone.output
    head = layers.Dense(1, activation='sigmoid')(features)
    temp_model = models.Model(inputs=backbone.input, outputs=head)
    temp_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    temp_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=26, batch_size=64, verbose=1)
    backbone.save(save_path)
    print(f"Backbone saved to {save_path}")

    feature_extractor = models.Model(inputs=backbone.input, outputs=backbone.output)
    train_feats = feature_extractor.predict(X_train)
    val_feats = feature_extractor.predict(X_val)

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(train_feats, y_train)
    joblib.dump(rf, 'branch1_rf.joblib')
    print("Branch 1 RF saved to branch1_rf.joblib")
    return feature_extractor, rf

In [ ]:
def train_branch2(X_train, y_train, X_val, y_val, input_shape, save_path='best_2D-trained_model.keras'):
    print("===== Branch 2 (Spatial) =====")
    backbone = build_spatial_feature_extractor(input_shape)
    features = backbone.output
    head = layers.Dense(1, activation='sigmoid')(features)
    temp_model = models.Model(inputs=backbone.input, outputs=head)
    temp_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    temp_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=26, batch_size=32, verbose=1)
    backbone.save(save_path)
    print(f"Backbone saved to {save_path}")

    feature_extractor = models.Model(inputs=backbone.input, outputs=backbone.output)
    train_feats = feature_extractor.predict(X_train)
    val_feats = feature_extractor.predict(X_val)

    xgb_clf = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1,
                                use_label_encoder=False, eval_metric='logloss')
    xgb_clf.fit(train_feats, y_train)
    joblib.dump(xgb_clf, 'branch2_xgb.joblib')
    print("Branch 2 XGB saved to branch2_xgb.joblib")
    return feature_extractor, xgb_clf

In [ ]:
def train_branch3(X_train, y_train, X_val, y_val):
    print("===== Branch 3 (BLS) =====")
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    joblib.dump(rf, 'branch3_rf.joblib')

    xgb_clf = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1,
                                use_label_encoder=False, eval_metric='logloss')
    xgb_clf.fit(X_train, y_train)
    joblib.dump(xgb_clf, 'branch3_xgb.joblib')
    print("Branch 3 models saved to branch3_rf.joblib and branch3_xgb.joblib")
    return rf, xgb_clf

In [ ]:
def train_meta_classifier(P1_val, C1_val, P2_val, C2_val, P3_val, C3_val, y_val):
    print("===== Stacking =====")
    X_meta = np.column_stack([P1_val, C1_val, P2_val, C2_val, P3_val, C3_val])
    meta_model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.05,
                                   use_label_encoder=False, eval_metric='logloss')
    meta_model.fit(X_meta, y_val, eval_set=[(X_meta, y_val)], early_stopping_rounds=50, verbose=False)
    joblib.dump(meta_model, 'meta_xgb.joblib')
    print("Meta-classifier saved to meta_xgb.joblib")
    return meta_model

In [ ]:
def main():
    # Load data
    light_curves, tpfs, labels = load_data()

    # Extract BLS features
    bls_features = extract_bls_features(light_curves, time=None, cadence=1.0)

    # Preprocess
    print("Preprocessing...")
    light_curves = preprocess_light_curves(light_curves)
    tpfs = preprocess_tpfs(tpfs)

    bls_scaler = StandardScaler()
    bls_features = bls_scaler.fit_transform(bls_features)
    joblib.dump(bls_scaler, 'bls_scaler.joblib')
    print("BLS scaler saved to bls_scaler.joblib")

    # Train/Val split
    X_lc_train, X_lc_val, X_tpf_train, X_tpf_val, \
    X_bls_train, X_bls_val, y_train, y_val = train_test_split(
        light_curves, tpfs, bls_features, labels,
        test_size=0.2, random_state=42, stratify=labels
    )

    # Branch 1
    feat_ext1, clf1 = train_branch1(
        X_lc_train, y_train, X_lc_val, y_val,
        input_shape=light_curves.shape[1:]
    )
    val_feats1 = feat_ext1.predict(X_lc_val)
    P1_val = clf1.predict_proba(val_feats1)[:, 1]
    C1_val = P1_val

    # Branch 2
    feat_ext2, clf2 = train_branch2(
        X_tpf_train, y_train, X_tpf_val, y_val,
        input_shape=tpfs.shape[1:]
    )
    val_feats2 = feat_ext2.predict(X_tpf_val)
    P2_val = clf2.predict_proba(val_feats2)[:, 1]
    C2_val = P2_val

    # Branch 3
    clf3_rf, clf3_xgb = train_branch3(X_bls_train, y_train, X_bls_val, y_val)
    P3_rf = clf3_rf.predict_proba(X_bls_val)[:, 1]
    P3_xgb = clf3_xgb.predict_proba(X_bls_val)[:, 1]
    P3_val = (P3_rf + P3_xgb) / 2.0
    C3_val = P3_val

    # Meta
    train_meta_classifier(P1_val, C1_val, P2_val, C2_val, P3_val, C3_val, y_val)

    print("\nAll models saved successfully!")
    print("Files created:")
    print("  best_1d_trained_model.keras")
    print("  best_2D-trained_model.keras")
    print("  branch1_rf.joblib")
    print("  branch2_xgb.joblib")
    print("  branch3_rf.joblib")
    print("  branch3_xgb.joblib")
    print("  meta_xgb.joblib")
    print("  bls_scaler.joblib")

In [ ]:
if __name__ == "__main__":
    main()